# Splunk SOAR: container disposition

Shows whether a container is open or closed and its final disposition (the `Disposition` custom field, e.g. true positive / benign positive).

The `ph-auth-token` is read from `SOAR2_TOKEN` in the repo-root `.env` file (git-ignored; see `../.env.example`).

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
import pandas as pd
import requests

SERVER = "https://prod-soar2.mdr.tekstream.com"  # SOAR 1
load_dotenv(find_dotenv(usecwd=True))
TOKEN = os.environ.get("SOAR2_TOKEN", "")  # from .env

CONTAINER_IDS = [247448]

assert TOKEN, "set SOAR2_TOKEN in .env (see ../.env.example)"
session = requests.Session()
session.headers.update({"ph-auth-token": TOKEN, "Content-Type": "application/json"})


def get(path, params=None):
    r = session.get(f"{SERVER}{path}", params=params, allow_redirects=False, timeout=60)
    r.raise_for_status()
    return r.json()

In [ ]:
# Status name -> status type (new / open / resolved); resolved = closed
statuses = get("/rest/container_status", {"page_size": 0})["data"]
STATUS_TYPE = {s["name"]: s["status_type"] for s in statuses}

rows = []
for cid in CONTAINER_IDS:
    c = get(f"/rest/container/{cid}")
    custom = c.get("custom_fields") or {}
    rows.append({
        "container_id": cid,
        "name": c.get("name"),
        "status": c.get("status"),
        "open_or_closed": "closed" if STATUS_TYPE.get(c.get("status")) == "resolved" else STATUS_TYPE.get(c.get("status")),
        "disposition": custom.get("Disposition"),
        "sa_decision": custom.get("SA Decision"),
        "close_time": c.get("close_time"),
    })

pd.DataFrame(rows).set_index("container_id").T